In [1]:
import os
from PIL import Image, ImageDraw

# 1. Ensure the directory exists
os.makedirs("images", exist_ok=True)

# 2. Define the path for our target test image
target_image_path = os.path.join("images", "pcb_missing_component.jpg")

# 3. Create a real 400x400 red square image so it contains valid pixel data
img = Image.new("RGB", (400, 400), color="red")
draw = ImageDraw.Draw(img)

# 4. Draw text inside it so LLaVA has structural information to look at
draw.text((20, 180), "🔴 DEFECT: Missing Component\nLocation: Top Left R1", fill="white")

# 5. Save it, overwriting the empty 0-byte placeholder
img.save(target_image_path)
print(f"✅ Generated a valid image file at: {target_image_path} ({os.path.getsize(target_image_path)} bytes)")

✅ Generated a valid image file at: images\pcb_missing_component.jpg (4944 bytes)


In [2]:
import os
import ollama

# Set paths matching your folder structure
IMAGE_DIR = "images"
DOCS_DIR = "docs"
KNOWLEDGE_BASE_FILE = os.path.join(DOCS_DIR, "pcb_inspection_guide.txt")

# Verify directories exist
print(f"Checking images folder: {os.path.exists(IMAGE_DIR)}")
print(f"Checking knowledge guide: {os.path.exists(KNOWLEDGE_BASE_FILE)}")

Checking images folder: True
Checking knowledge guide: True


In [3]:
def inspect_pcb(image_path):
    # Convert relative path to a full Windows absolute path string for Ollama
    absolute_path = os.path.abspath(image_path)
    print(f"🔍 Analyzing absolute path: {absolute_path} using LLaVA...")
    
    prompt = """
    You are an IPC Certified PCB Quality Inspector.
    Analyze the uploaded PCB image. Your task is to inspect the PCB for manufacturing defects.
    
    Return the result exactly in this text format:
    PCB Status: [PASS or FAIL]
    Detected Components: [List main components]
    Detected Defects: [Name the primary defect, e.g., Missing Component, Solder Bridge, Misalignment, Short Circuit, None]
    Defect Location: [e.g., Top Left, Center, Bottom Right, None]
    Severity: [Low/Medium/High]
    Confidence: [Low/Medium/High]
    
    Do not guess. If uncertain, state that manual inspection is recommended.
    """
    
    # Pass the absolute string path directly into the images framework array
    response = ollama.generate(
        model='llava', 
        prompt=prompt,
        images=[absolute_path]
    )
    
    return response['response']

In [4]:
def query_knowledge_base(defect_name):
    if not os.path.exists(KNOWLEDGE_BASE_FILE):
        return "⚠️ Error: Reference manual not found."
        
    with open(KNOWLEDGE_BASE_FILE, 'r') as f:
        content = f.read()
        
    # Split manual sections by tag
    sections = content.split("[DEFECT: ")
    
    for section in sections:
        # Check if the text matches the model's detected defect name
        if defect_name.lower() in section.lower():
            return f"🔧 Standard IPC Repair Guide for [{defect_name.strip()}]:\n" + section.strip()
            
    return f"ℹ️ No specific standard procedures found in manual for: {defect_name}."

In [7]:
def parse_defect_type(output_text):
    """Helper utility to extract the defect string out of the LLaVA response"""
    defect = "None"
    for line in output_text.split('\n'):
        if "Detected Defects:" in line:
            defect = line.split("Detected Defects:")[1].strip()
    return defect

def run_inspection_pipeline(image_filename):
    image_path = os.path.join(IMAGE_DIR, image_filename)
    
    # 1. Run Vision AI Assessment
    vision_result = inspect_pcb(image_path)
    
    # 2. Extract Defect Entity & Check Knowledge Base (RAG)
    detected_defect = parse_defect_type(vision_result)
    
    repair_guide = "N/A (PCB Passed Inspection)"
    if "none" not in detected_defect.lower() and detected_defect != "":
        repair_guide = query_knowledge_base(detected_defect)
        
    # 3. Format the Final QA Report
    final_report = f"""
==================================================
           PCB AI INSPECTION REPORT
==================================================
Target Image File: {image_filename}
--------------------------------------------------
[1. VISION INTERPRETATION RESULTS]
{vision_result}

--------------------------------------------------
[2. RAG GUIDELINES & REMEDIAL ACTIONS]
{repair_guide}
==================================================
"""
    # 4. Save report to disk -> Added explicit UTF-8 encoding here to handle emojis
    os.makedirs("reports", exist_ok=True)
    report_path = f"reports/report_{os.path.splitext(image_filename)[0]}.txt"
    with open(report_path, "w", encoding="utf-8") as f:
        f.write(final_report)
        
    print(f"\n📁 Report successfully logged to: {report_path}")
    print(final_report)

In [8]:
run_inspection_pipeline("pcb_missing_component.jpg")

🔍 Analyzing absolute path: c:\Users\Shreyash\OneDrive\Desktop\genaiproj\images\pcb_missing_component.jpg using LLaVA...

📁 Report successfully logged to: reports/report_pcb_missing_component.txt

           PCB AI INSPECTION REPORT
Target Image File: pcb_missing_component.jpg
--------------------------------------------------
[1. VISION INTERPRETATION RESULTS]
 PCB Status: FAIL
     Detected Components: [Vcc, R1, C1, R2, LED]
     Detected Defects: [Missing Component, Short Circuit]
     Defect Location: [Center Bottom, Top Left Corner]
     Severity: [High]
     Confidence: [Medium] 

--------------------------------------------------
[2. RAG GUIDELINES & REMEDIAL ACTIONS]
ℹ️ No specific standard procedures found in manual for: [Missing Component, Short Circuit].

